In [ ]:
import pandas as pd
import numpy as np
import sys
import io
import random
import time
import os

print("ETAPA 1: CARREGANDO ARQUIVOS\n")

url_parentesco_produtos = "https://raw.githubusercontent.com/devThalys/antcolonyoptmization/refs/heads/main/parentesco_produtos.csv"
url_pais_indices = "https://raw.githubusercontent.com/devThalys/antcolonyoptmization/refs/heads/main/pais_indice.csv"
url_pais_indices_modificados = "https://raw.githubusercontent.com/devThalys/antcolonyoptmization/refs/heads/main/pais_indice_modificado.csv"

df_coancestralidade = pd.read_csv(url_parentesco_produtos, sep=',')
df_coancestralidade.columns = df_coancestralidade.columns.str.strip().str.lower()
if len(df_coancestralidade.columns) == 3:
    df_coancestralidade.columns = ['animal1', 'animal2', 'coef']
print("Arquivo 'parentesco_produtos.csv' carregado e padronizado.")

print("\n" + "="*40)
print("     SELEÇÃO DO ARQUIVO DE ÍNDICES")
print("="*40)
print("1. Arquivo Original (GitHub)")
print("2. Arquivo Modificado (GitHub)")
print("3. Selecionar Arquivo Novo (Upload/Janela)")

arquivo_selecionado = ""
usar_io_bytes = False

while True:
    opcao_arquivo = input("Digite o número da opção (1, 2 ou 3): ")

    if opcao_arquivo == '1':
        arquivo_selecionado = url_pais_indices
        print("-> Selecionado: Arquivo Original (GitHub)")
        break
    elif opcao_arquivo == '2':
        arquivo_selecionado = url_pais_indices_modificados
        print("-> Selecionado: Arquivo Modificado (GitHub)")
        break
    elif opcao_arquivo == '3':
        try:
            from google.colab import files
            print("\nPor favor, faça o upload do arquivo usando o botão abaixo:")
            uploaded = files.upload()

            if not uploaded:
                print("Nenhum arquivo foi selecionado. Tente novamente.")
                continue

            nome_arquivo = next(iter(uploaded))
            arquivo_selecionado = io.BytesIO(uploaded[nome_arquivo])
            usar_io_bytes = True
            print(f"-> Arquivo recebido: {nome_arquivo}")
            break

        except ImportError:
            try:
                import tkinter as tk
                from tkinter import filedialog

                root = tk.Tk()
                root.withdraw()

                print("\nAbrindo janela de seleção de arquivo...")
                caminho = filedialog.askopenfilename(title="Selecione o arquivo CSV", filetypes=[("CSV files", "*.csv")])

                if caminho:
                    arquivo_selecionado = caminho
                    print(f"-> Selecionado: {caminho}")
                    break
                else:
                    print("Nenhum arquivo selecionado. Tente novamente.")
            except Exception as e:
                print(f"\nNão foi possível abrir a janela (Erro: {e}).")
                path_input = input("Digite o caminho do arquivo manualmente: ").strip().strip('"').strip("'")
                if os.path.exists(path_input):
                    arquivo_selecionado = path_input
                    break
                else:
                    print("Arquivo não encontrado.")
    else:
        print("Opção inválida! Digite 1, 2 ou 3.")

try:
    df_indice = pd.read_csv(arquivo_selecionado, sep=',')
    print(f"Arquivo carregado com sucesso!")
except Exception as e:
    print(f"\nERRO CRÍTICO ao ler o arquivo: {e}")
    sys.exit()

df_indice.columns = [col.strip().lower() for col in df_indice.columns]

if len(df_indice.columns) >= 2:
    if 'animal' in df_indice.columns[0] and 'indice' in df_indice.columns[1]:
        df_indice.columns = ['animal_id', 'indice']
    else:
        cols = df_indice.columns
        df_indice.rename(columns={cols[0]: 'animal_id', cols[1]: 'indice'}, inplace=True)

print("Arquivo de índices padronizado.")


print("\nETAPA 2: MAPEANDO MACHOS (M) E FÊMEAS (F)\n")

macho_ids_set = set()
femea_ids_set = set()

def mapear_animal(animal_str):
    try:
        macho_str, femea_str = str(animal_str).split('_')
        macho_ids_set.add(int(macho_str))
        femea_ids_set.add(int(femea_str))
    except (ValueError, TypeError):
        pass

df_coancestralidade['animal1'].apply(mapear_animal)
df_coancestralidade['animal2'].apply(mapear_animal)

lista_machos = sorted(list(macho_ids_set))
lista_femeas = sorted(list(femea_ids_set))
num_femeas = len(lista_femeas)
num_machos = len(lista_machos)

print(f"Total de Fêmeas únicas |F|: {num_femeas}")
print(f"Total de Machos únicos |M|: {num_machos}")

print("ETAPA 3: PREPARANDO DATAFRAMES\n")

df_indice['animal_id'] = pd.to_numeric(df_indice['animal_id'], errors='coerce')
df_indice.dropna(subset=['animal_id'], inplace=True)
df_indice['animal_id'] = df_indice['animal_id'].astype(int)

df_machos = pd.DataFrame(lista_machos, columns=['animal_id'])
df_femeas = pd.DataFrame(lista_femeas, columns=['animal_id'])

df_machos = pd.merge(df_machos, df_indice, on='animal_id', how='left')
df_femeas = pd.merge(df_femeas, df_indice, on='animal_id', how='left')

df_femeas['indice'] = df_femeas['indice'].fillna(0)
df_machos['indice'] = df_machos['indice'].fillna(0)
print("Dados pré-processados com sucesso")

print("ETAPA 4: CONSTRUINDO MATRIZ DE HEURÍSTICA (Eta)\n")

indices_femeas = df_femeas['indice'].values
indices_machos = df_machos['indice'].values

soma_indices = indices_femeas.reshape(-1, 1) + indices_machos.reshape(1, -1)
matriz_indices_reais = soma_indices / 2

min_val = np.min(matriz_indices_reais)
if min_val <= 0:
    print(f"Detectados valores negativos/zero ({min_val}). \nNormalizando matriz para ACO...\n")
    matriz_heuristica_eta = matriz_indices_reais + abs(min_val) + 1.0
else:
    matriz_heuristica_eta = matriz_indices_reais

print(f"Matriz de Heurística pronta.\nShape: {matriz_heuristica_eta.shape}\n")

print("\nETAPA 5: INDEXANDO COANCESTRALIDADE (HASHMAP)\n")

coancestralidade_dict = {}
termos_a = df_coancestralidade['animal1'].values
termos_b = df_coancestralidade['animal2'].values
coefs = df_coancestralidade['coef'].values

def parse_key(termo):
    m, f = termo.split('_')
    return (int(m), int(f))

print("Criando dicionário...")
for i in range(len(coefs)):
    key_a = parse_key(termos_a[i])
    key_b = parse_key(termos_b[i])
    val = coefs[i]
    coancestralidade_dict[(key_a, key_b)] = val
    coancestralidade_dict[(key_b, key_a)] = val

print(f"Dicionário criado com {len(coancestralidade_dict)} entradas.")

print("\n" + "="*40)
print("          MENU DE CONFIGURAÇÃO ACO")
print("="*40)

MAX_USO_MACHO = 0

while True:
    print("\nEscolha o numero maximo de usos do macho:")
    print("1. 12 usos máximos")
    print("2. 18 usos máximos")
    print("3. 20 usos máximos")
    print("4. Escolher manualmente")

    opcao = input("Digite o número da opção (1, 2, 3 ou 4): ")

    if opcao == '1':
        MAX_USO_MACHO = 12
        break
    elif opcao == '2':
        MAX_USO_MACHO = 18
        break
    elif opcao == '3':
        MAX_USO_MACHO = 20
        break
    elif opcao == '4':
        MAX_USO_MACHO = int(input("Digite o número máximo de usos do macho: "))
        break
    else:
        print("Opção inválida! Por favor, digite 1, 2, 3 ou 4.")

print(f"\nConfiguração selecionada:\nMáximo de {MAX_USO_MACHO} usos por macho.")

print("ETAPA 6: ALGORITMO DE COLÔNIA DE FORMIGAS (ACO)\n")

NUM_FORMIGAS = 20
NUM_ITERACOES = 50
ALPHA = 1.0
BETA = 3.0
EVAPORATION = 0.1
Q = 100

tau = np.ones((num_femeas, num_machos)) * 1.0

best_solution = None
best_cost = float('inf')
best_index_sum = -float('inf')
history_cost = []

print("Número de formigas: ",NUM_FORMIGAS)
print("Número de iterações: ",NUM_ITERACOES)
print("Alpha: ", ALPHA)
print("Beta: ", BETA)
print("Evaporation: ",EVAPORATION)
print("Q: ",Q)

print(f"\nIniciando ACO...")
start_time = time.time()

for it in range(NUM_ITERACOES):
    solutions_in_iter = []

    for ant in range(NUM_FORMIGAS):
        caminho_formiga = []
        uso_machos = {m_idx: 0 for m_idx in range(num_machos)}
        valid_path = True

        for i_f in range(num_femeas):
            probabilidades = []
            candidatos_validos = []

            for i_m in range(num_machos):
                if uso_machos[i_m] >= MAX_USO_MACHO:
                    prob = 0.0
                else:
                    t = tau[i_f, i_m] ** ALPHA
                    n = matriz_heuristica_eta[i_f, i_m] ** BETA
                    prob = t * n
                    candidatos_validos.append(i_m)
                probabilidades.append(prob)

            denominador = sum(probabilidades)

            if denominador > 0:
                probabilidades = [p / denominador for p in probabilidades]
                probabilidades = [(p / denominador) + 1e-6 for p in probabilidades]
                soma = sum(probabilidades)
                probabilidades = [p / soma for p in probabilidades]
                escolha_macho_idx = np.random.choice(range(num_machos), p=probabilidades)
            else:
                if not candidatos_validos:
                    valid_path = False
                    break
                escolha_macho_idx = random.choice(candidatos_validos)

            caminho_formiga.append((i_f, escolha_macho_idx))
            uso_machos[escolha_macho_idx] += 1

        if not valid_path:
            continue

        cost_coancestralidade = 0.0
        sum_indice_genetico = 0.0

        pares_reais = []
        for i_f, i_m in caminho_formiga:
            id_f = lista_femeas[i_f]
            id_m = lista_machos[i_m]
            pares_reais.append((id_m, id_f))
            sum_indice_genetico += matriz_indices_reais[i_f, i_m]

        n_pares = len(pares_reais)
        for i in range(n_pares):
            for j in range(i + 1, n_pares):
                par_a = pares_reais[i]
                par_b = pares_reais[j]
                cost_coancestralidade += coancestralidade_dict.get((par_a, par_b), 0.0)

        solutions_in_iter.append({
            'caminho': caminho_formiga,
            'custo': cost_coancestralidade,
            'indice_total': sum_indice_genetico
        })

        if cost_coancestralidade < best_cost:
            best_cost = cost_coancestralidade
            best_solution = caminho_formiga
            best_index_sum = sum_indice_genetico
        elif cost_coancestralidade == best_cost:
            if sum_indice_genetico > best_index_sum:
                best_solution = caminho_formiga
                best_index_sum = sum_indice_genetico

    tau *= (1 - EVAPORATION)

    for sol in solutions_in_iter:
        custo = sol['custo']
        if custo <= 0.0001: custo = 0.0001

        delta = Q / custo

        for i_f, i_m in sol['caminho']:
            tau[i_f, i_m] += delta

    print(f"Iteração {it}: Melhor Custo (Coancestralidade) = {best_cost:.4f} | Soma Índices = {best_index_sum:.2f}")

end_time = time.time()
print(f"\nTempo total: {end_time - start_time:.2f}s")

print("ETAPA 7: RESULTADOS E RELATÓRIO\n")

if best_solution:
    resultado_final = []
    ids_machos_usados = []

    print(f"{'Fêmea ID':<10} | {'Macho ID':<15} | {'Índice Real'}")
    print("-" * 45)

    indices_prole = []

    for i_f, i_m in best_solution:
        id_f = lista_femeas[i_f]
        id_m = lista_machos[i_m]
        val_indice = matriz_indices_reais[i_f, i_m]

        resultado_final.append({'femea': id_f, 'macho': id_m, 'indice': val_indice})
        ids_machos_usados.append(id_m)
        indices_prole.append(val_indice)

        if len(resultado_final) <= 10:
            print(f"{id_f:<10} | {id_m:<15} | {val_indice:.4f}")

    if len(resultado_final) > 10:
        print(f"... e mais {len(resultado_final)-10} pares.")

    contagem_machos = pd.Series(ids_machos_usados).value_counts()

    print("\n--- ESTATÍSTICAS FINAIS ---")
    print(f"Configuração Escolhida: {MAX_USO_MACHO} usos máximos")
    print(f"Coancestralidade (Custo Min): {best_cost:.4f}")
    print(f"Soma Índices Genéticos (Max): {best_index_sum:.4f}")
    print(f"Média Índice Prole: {np.mean(indices_prole):.4f}")
    print(f"Macho mais utilizado: {contagem_machos.max()} vezes (Limite: {MAX_USO_MACHO})")

    if contagem_machos.max() <= MAX_USO_MACHO:
        print("STATUS: Restrição respeitada!")
    else:
        print("STATUS: Restrição VIOLADA!")

else:
    print("FALHA CRÍTICA: Nenhuma solução encontrada.")

ETAPA 1: CARREGANDO ARQUIVOS

Arquivo 'parentesco_produtos.csv' carregado e padronizado.

     SELEÇÃO DO ARQUIVO DE ÍNDICES
1. Arquivo Original (GitHub)
2. Arquivo Modificado (GitHub)
3. Selecionar Arquivo Novo (Upload/Janela)
Digite o número da opção (1, 2 ou 3): 2
-> Selecionado: Arquivo Modificado (GitHub)
Arquivo carregado com sucesso!
Arquivo de índices padronizado.

ETAPA 2: MAPEANDO MACHOS (M) E FÊMEAS (F)

Total de Fêmeas únicas |F|: 60
Total de Machos únicos |M|: 5
ETAPA 3: PREPARANDO DATAFRAMES

Dados pré-processados com sucesso
ETAPA 4: CONSTRUINDO MATRIZ DE HEURÍSTICA (Eta)

Detectados valores negativos/zero (0.0). 
Normalizando matriz para ACO...

Matriz de Heurística pronta.
Shape: (60, 5)


ETAPA 5: INDEXANDO COANCESTRALIDADE (HASHMAP)

Criando dicionário...
Dicionário criado com 89700 entradas.

          MENU DE CONFIGURAÇÃO ACO

Escolha o numero maximo de usos do macho:
1. 12 usos máximos
2. 18 usos máximos
3. 20 usos máximos
4. Escolher manualmente
Digite o número d